# Week 9 · Day 5 — Checkpoint 9: the scorecard review for the risk committee

Eight tasks in three parts — **A** performance, **B** stability, **C** the cut-off. For each: code cell(s) with the result printed, then a Markdown cell with **two sentences of interpretation** the risk committee can read: what the number says, with its n, and what to do with it.
Part D, the one-page memo, goes into `memo.md`, not here.
Hand this notebook in **executed** (Kernel → Restart & Run All, then save) with the 📎 box at the end of the Week 9 · Day 5 page.

**Bad** = a loan whose status is `default` or `written_off`. Data as of 2026-10-01.

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
# this notebook lives in data/python/week9/, so "../.." is the data/ folder
DATA = Path("../..")
AS_OF = pd.Timestamp("2026-10-01")
con = sqlite3.connect(DATA / "raw" / "nur.db")
# one seeded generator: the same "random" numbers on every run
rng = np.random.default_rng(2026)

In [2]:
# The frames of the checkpoint (built for you; run, do not edit)
# Week 8 Day 2's assumptions (ours): funding 18 % a year, LGD 60 %,
# acquisition per loan and servicing per month by channel
COF, LGD = 0.18, 0.60
ACQ = {"app": 60_000, "branch": 450_000, "partner": 100_000}
SERV = {"app": 8_000, "branch": 25_000, "partner": 8_000}
# the six score bands of the week
EDGES = [0, 560, 600, 640, 680, 720, 10_000]
LABELS = ["<560", "560-599", "600-639", "640-679", "680-719", "720+"]

# loans: one row per loan — the score at the door, the outcome and
# Week 8 Day 2's per-loan economics (interest over the schedule,
# balance-months for the cost of funds, EAD = the balance owed at the
# first unpaid instalment)
loans = pd.read_sql("""
WITH pay AS (
  SELECT installment_id, paid_at, amount FROM (
    SELECT installment_id, paid_at, amount,
           ROW_NUMBER() OVER (PARTITION BY installment_id, paid_at, amount ORDER BY payment_id) AS rn
    FROM payments) WHERE rn = 1),
sched AS (
  SELECT i.loan_id, i.installment_no, i.installment_id, i.due_date, i.total_due, i.interest_due,
         l.principal - COALESCE(SUM(i.principal_due) OVER (PARTITION BY i.loan_id ORDER BY i.installment_no
                       ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS bal_before
  FROM installments i JOIN loans l ON l.loan_id = i.loan_id),
unpaid AS (
  SELECT s.installment_id, s.loan_id, s.installment_no
  FROM sched s LEFT JOIN pay p ON p.installment_id = s.installment_id AND p.paid_at <= '2026-10-01'
  WHERE s.due_date <= '2026-10-01'
  GROUP BY s.installment_id, s.loan_id, s.installment_no, s.total_due
  HAVING COALESCE(SUM(p.amount), 0) < s.total_due),
first_unpaid AS (SELECT loan_id, MIN(installment_no) AS k FROM unpaid GROUP BY loan_id)
SELECT l.loan_id, pr.code AS product, pr.channel, l.risk_grade, l.status,
       l.disbursed_at, l.principal, l.term_months AS term, a.credit_score,
       SUM(s.interest_due) AS interest, SUM(s.bal_before) AS bal_months,
       MAX(CASE WHEN s.installment_no = f.k THEN s.bal_before END) AS ead
FROM loans l
JOIN products pr     ON pr.product_id = l.product_id
JOIN applications a  ON a.application_id = l.application_id
JOIN sched s         ON s.loan_id = l.loan_id
LEFT JOIN first_unpaid f ON f.loan_id = l.loan_id
GROUP BY l.loan_id, pr.code, pr.channel, l.risk_grade, l.status, l.disbursed_at,
         l.principal, l.term_months, a.credit_score
ORDER BY l.loan_id""", con, parse_dates=["disbursed_at"])
loans["bad"] = loans["status"].isin(["default", "written_off"]).astype(int)
loans["year"] = loans["disbursed_at"].dt.year
loans["band"] = pd.cut(loans["credit_score"], EDGES, right=False, labels=LABELS)
# sc: the loans with a score at the door — the scorecard's own book
sc = loans.dropna(subset=["credit_score"]).copy()

# apps: one row per application, with the applicant's score and its band
apps = pd.read_sql("""
SELECT a.application_id, a.applied_at, a.channel, p.code AS product,
       a.credit_score, a.status
FROM applications a JOIN products p ON p.product_id = a.product_id""", con, parse_dates=["applied_at"])
apps["year"] = apps["applied_at"].dt.year
apps["band"] = pd.cut(apps["credit_score"], EDGES, right=False, labels=LABELS)

assert len(loans) == 7828 and loans["bad"].sum() == 234
assert len(sc) == 7615 and sc["bad"].sum() == 223 and len(apps) == 14000
print(f"loans {len(loans)} · bad {loans['bad'].sum()} ({loans['bad'].mean():.2%}) · "
      f"with a score {len(sc)} ({sc['bad'].sum()} bad) · applications {len(apps)}")

loans 7828 · bad 234 (2.99%) · with a score 7615 (223 bad) · applications 14000


## Part A — Performance: does the score still rank?

### A1. AUC, Gini and KS of the credit score — all loans

On `sc` (7 615 loans, 223 bad): the AUC (good–bad pairs where the good loan has the higher score, ties half), the Gini = 2 · AUC − 1, and the KS with the score where it happens. How good is the ranking against the usual 0.4–0.6?

In [3]:
# your code

**Interpretation (two sentences):** …

### A2. By grade — four steps of the same score

Each grade's loans, bads, bad rate and score range. Then the grade's own Gini (A = 4, B = 3, C = 2, D = 1, on the same 7 615 loans) and the score's Gini inside each grade. Where does the ranking power live?

In [4]:
# your code

**Interpretation (two sentences):** …

### A3. Out of time — older loans against 2026

For 2024–2025 and for 2026 (year of disbursement): loans, bads, bad rate, AUC, Gini, KS; a bootstrap 95 % interval of each Gini (1 000 resamples, `np.random.default_rng(2026)`); the 2026 bads by month. Has the score got worse, or can the data not tell yet?

In [5]:
# your code

**Interpretation (two sentences):** …

## Part B — Stability: are we scoring the same people?

### B1. PSI of applicants' scores — 2026 against 2024

Applications with a score, by `band`: each year's shares, each band's part (actual − expected) × ln(actual ÷ expected) with 2024 as expected, and the PSI for 2025 and for 2026. Read it on the traffic light (below 0.1 / 0.1–0.25 / above 0.25).

In [6]:
# your code

**Interpretation (two sentences):** …

### B2. Two more views — the product mix, and the approved loans

(1) The CSI of the applicants' product mix, 2026 against 2024. (2) The same PSI on the approved loans (`sc`, by year of disbursement), with both years' band shares. Why is the loans' PSI larger, and which one belongs in the committee's stability line?

In [7]:
# your code

**Interpretation (two sentences):** …

## Part C — The cut-off: where should the line sit?

### C1. The trade-off — loans kept against bad rate

For the cut-offs 560, 600, 620, 640, 660 (approve score ≥ c) on the 7 615 scored loans: the share kept, their bad rate, the bads avoided. SQL (`pd.read_sql`) or pandas.

In [8]:
# your code

**Interpretation (two sentences):** …

### C2. Profit by cut-off — Micro, the app's own decision

Micro loans with a score (2 598, 73 bad). Each loan's contribution with Week 8 Day 2's assumptions (`COF`, `LGD`, `ACQ`, `SERV`): a good loan earns its interest; a bad loan earns none and loses LGD × EAD; both pay the cost of funds (balance-months × COF ÷ 12), acquisition and servicing (per month × term). Then for no cut-off and every 10 points from 540 to 720: share kept, bad rate, bads avoided, total contribution in million soum. Which cut-off earns most?

In [9]:
# your code

**Interpretation (two sentences):** …

### C3. Who the new cut-off refuses — the swap set and the recommendation

The swap-out set of a 660 cut-off (the Micro loans it would have refused): loans, bads, bad rate and contribution by grade and in total, against Week 8 Day 2's Micro break-even PD (4.23 % first loan, 7.23 % repeat loan). Then your recommendation in three sentences: the cut-off, what it gains and costs, and how you would roll it out.

In [10]:
# your code

**Interpretation (two sentences):** …

## Part D — the memo

Write the one-page memo for the risk committee in `memo.md` (the skeleton is on the Week 9 · Day 5 page) and paste it into the 📎 box there. Every number in it should be printed somewhere above.